In [5]:
from pathlib import Path
import importlib
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_dir = project_root / "src"
src_dir.mkdir(parents=True, exist_ok=True)
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

sys.modules.pop("analisis_riesgo", None)
importlib.invalidate_caches()

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError:
    XGBRegressor = None

from analisis_riesgo import *

if "DATA_RAW" not in globals() or DATA_RAW is None:
    DATA_RAW = next(
        (p for p in [project_root / "data" / "raw", project_root.parent / "data" / "raw"] if p.exists()),
        project_root / "data" / "raw",
    )

if "candidatos" not in globals() or not candidatos:
    candidatos = [str(project_root), str(project_root.parent), str(Path.cwd())]

# DATA_RAW y candidatos ya están definidos en el notebook o se calculan por defecto.
data_raw = Path(DATA_RAW)
if not data_raw.exists():
    data_raw = next(
        (Path(p) / "data" / "raw" for p in candidatos if (Path(p) / "data" / "raw").exists()),
        data_raw,
    )

project_root = data_raw.parent.parent
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

archivos, encabezados, variables_seleccionadas = inspeccionar_csv(data_raw)
df, riesgo_parroquia, riesgo_canton, metricas, predicciones = ejecutar_pipeline(
    data_raw,
    results_dir,
    project_root / "data" / "processed" / "incidentes_limpios.csv",
)

display(riesgo_parroquia.head(20))
display(metricas)


Archivos CSV encontrados: 59

C:\Users\PAULA\Documents\IntegradorG3\data\raw\data2021\emergencias_agosto_2021.csv
  Variables: ['\ufeffFecha', 'provincia', 'Canton', 'Cod_Parroquia', 'Parroquia', 'Servicio', 'Subtipo']

C:\Users\PAULA\Documents\IntegradorG3\data\raw\data2021\emergencias_diciembre_2021.csv
  Variables: ['\ufeffFecha', 'provincia', 'Canton', 'Cod_Parroquia', 'Parroquia', 'Servicio', 'Subtipo']

C:\Users\PAULA\Documents\IntegradorG3\data\raw\data2021\emergencias_julio_2021.csv
  Variables: ['\ufeffFecha', 'provincia', 'Canton', 'Cod_Parroquia', 'Parroquia', 'Servicio', 'Subtipo']

C:\Users\PAULA\Documents\IntegradorG3\data\raw\data2021\emergencias_noviembre_2021.csv
  Variables: ['\ufeffFecha', 'provincia', 'Canton', 'Cod_Parroquia', 'Parroquia', 'Servicio', 'Subtipo']

C:\Users\PAULA\Documents\IntegradorG3\data\raw\data2021\emergencias_octubre_2021.csv
  Variables: ['\ufeffFecha', 'provincia', 'Canton', 'Cod_Parroquia', 'Parroquia', 'Servicio', 'Subtipo', 'Unnamed: 7']



,canton,parroquia,incidentes_totales,promedio_mensual,variabilidad_mensual,pico_mensual,cluster,nivel_cluster
23,AMBATO,"AMBATO, CABECERA CANTONAL Y CAPITAL PROVINCIAL",1796,30.440678,0.836066,31,1,4
38,AMBATO,SANTA ROSA,1796,30.440678,0.836066,31,1,4
41,ANTONIO ANTE,"ATUNTAQUI, CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4
79,BABAHOYO,"BABAHOYO, CABECERA CANTONAL Y CAPITAL PROVINCIAL",1796,30.440678,0.836066,31,1,4
87,BALZAR,"BALZAR, CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4
88,BAÑOS DE AGUA SANTA,"BAÑOS DE AGUA SANTA, CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4
108,BUENA FE,"SAN JACINTO DE BUENA FÉ, CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4
121,CATAMAYO,"CATAMAYO (LA TOMA), CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4
128,CAYAMBE,"CAYAMBE, CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4
177,CHONE,"CHONE, CABECERA CANTONAL",1796,30.440678,0.836066,31,1,4


,modelo,MAE,RMSE,R2,periodos_prueba
0,Random Forest,5.150842,8.954885,0.434495,12
1,XGBoost,5.401019,9.254417,0.396032,12
